# 第二章 PyTorch：张量操作与自动微分

对应 Keras 版本：`Keras应用/第二章-神经网络的数学基础/`

**学习路线：**

| 阶段 | 章节 | 内容 | 关键词 |
|------|------|------|--------|
| **一、张量基础** | §1-§2 | 张量创建、切片、索引、操作 | Tensor, reshape, view, device |
| **二、自动微分** | §3-§4 | autograd、梯度计算、计算图 | requires_grad, backward, grad |
| **三、实战** | §5 | 纯张量实现 MNIST 分类 | 手动前向/反向传播 |

> 本笔记本使用 PyTorch，所有代码可直接运行。

In [1]:
import torch
import numpy as np

print(f"PyTorch: {torch.__version__}")
print(f"CUDA 可用: {torch.cuda.is_available()}")

PyTorch: 2.7.1+cu118
CUDA 可用: True


---
# 第一阶段：张量基础

对应 Keras 版本的 TensorFlow 张量概念。

## §1 张量（Tensor）是什么？

张量是深度学习中的核心数据结构 —— 本质上是一个**多维数组**。

| 张量阶数 | 数学名 | 示例 |
|----------|--------|------|
| 0 阶 | 标量 (Scalar) | `5` 一个数字 |
| 1 阶 | 向量 (Vector) | `[1, 2, 3]` 一组数字 |
| 2 阶 | 矩阵 (Matrix) | `[[1,2],[3,4]]` 表格 |
| 3 阶 | 3D 张量 | MNIST 图像数据集 `(60000, 28, 28)` |
| 4 阶 | 4D 张量 | 彩色图像批次 `(batch, height, width, channels)` |

**与 NumPy 的关系**：PyTorch Tensor 和 NumPy ndarray 非常相似，但 Tensor 可以在 GPU 上运行。

In [ ]:
# 1.1 创建张量

# 从列表/数组创建
x = torch.tensor([[1, 2], [3, 4]])
print(f"从列表创建:\n{x}")

# 零/一/随机张量
zeros = torch.zeros(2, 3)
ones = torch.ones(2, 3)
rand = torch.randn(2, 3)       # 标准正态分布
rand_int = torch.randint(0, 10, (2, 3))  # 随机整数

print(f"\nzeros:\n{zeros}")
print(f"\nones:\n{ones}")
print(f"\nrandn:\n{rand}")
print(f"\nrandint:\n{rand_int}")

In [ ]:
# 1.2 张量的属性
x = torch.randn(4, 3, 2)

print(f"形状 (shape): {x.shape}")
print(f"维度数 (ndim): {x.ndim}")
print(f"元素总数 (numel): {x.numel()}")
print(f"数据类型 (dtype): {x.dtype}")
print(f"所在设备 (device): {x.device}")

### 与 NumPy 的互操作

In [ ]:
# Tensor → NumPy
t = torch.randn(3, 3)
n = t.numpy()
print(f"Tensor → NumPy: type={type(n)}, shape={n.shape}")

# NumPy → Tensor
n2 = np.random.randn(3, 3).astype(np.float32)
t2 = torch.from_numpy(n2)
print(f"NumPy → Tensor: type={type(t2)}, shape={t2.shape}")

---
## §2 张量操作：切片、索引、形状变换

对应 Keras 版本中的数组切片示例。

In [ ]:
# 2.1 加载 MNIST 数据集（与 Keras 版本相同的数据）
from torchvision import datasets, transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True)

# 获取原始数据（未 transform 的 PIL 图像）
train_images = train_dataset.data.float() / 255.0  # 归一化到 [0, 1]
train_labels = train_dataset.targets

print(f"训练图像形状: {train_images.shape}")  # (60000, 28, 28)
print(f"训练标签数量: {len(train_labels)}")
print(f"第一个训练图像的标签: {train_labels[0]}")

In [ ]:
# 2.2 张量切片与索引（对应 Keras 版本的切片示例）

# 示例 1：选择第 10 到第 100 个数字（不包括第 100 个）
my_slice = train_images[10:100]
print(f"切片 1 形状 (10:100): {my_slice.shape}")  # (90, 28, 28)

# 示例 2：在所有图像的右下角选出 14x14 像素
my_slice_corner = train_images[:, 14:, 14:]
print(f"右下角切片形状: {my_slice_corner.shape}")  # (60000, 14, 14)

# 示例 3：使用负数索引在图像中心裁剪 14x14
my_slice_center = train_images[:, 7:-7, 7:-7]
print(f"中心裁剪切片形状: {my_slice_center.shape}")  # (60000, 14, 14)

# 示例 4：获取一个 batch
batch = train_images[:128]
print(f"batch 形状: {batch.shape}")  # (128, 28, 28)

In [ ]:
# 2.3 形状变换（reshape / view）

x = torch.randn(4, 3, 2)

# reshape：改变形状，不改变数据
y1 = x.reshape(4, 6)    # 合并后两维
y2 = x.reshape(-1, 6)   # -1 表示自动推断
y3 = x.view(24, 2)      # view 和 reshape 类似
y4 = x.flatten()        # 展平为一维

print(f"原始形状: {x.shape}")
print(f"reshape(4,6):   {y1.shape}")
print(f"reshape(-1,6):  {y2.shape}")
print(f"view(24,2):     {y3.shape}")
print(f"flatten():      {y4.shape}")

In [ ]:
# 2.4 维度操作

x = torch.randn(4, 3)

# squeeze：去掉维度为 1 的维度
a = torch.randn(1, 4, 1, 3)
print(f"原始: {a.shape}")
print(f"squeeze: {a.squeeze().shape}")       # (4, 3)
print(f"squeeze(0): {a.squeeze(0).shape}")   # (4, 1, 3)

# unsqueeze：在指定位置插入维度
b = torch.randn(4, 3)
print(f"\n原始: {b.shape}")
print(f"unsqueeze(0): {b.unsqueeze(0).shape}")  # (1, 4, 3)
print(f"unsqueeze(-1): {b.unsqueeze(-1).shape}")  # (4, 3, 1)

In [ ]:
# 2.5 拼接与堆叠

a = torch.randn(2, 3)
b = torch.randn(2, 3)

# cat：沿已有维度拼接
cat0 = torch.cat([a, b], dim=0)  # (4, 3)
cat1 = torch.cat([a, b], dim=1)  # (2, 6)

# stack：沿新维度堆叠
stack0 = torch.stack([a, b], dim=0)  # (2, 2, 3)

print(f"cat dim=0: {cat0.shape}")
print(f"cat dim=1: {cat1.shape}")
print(f"stack dim=0: {stack0.shape}")

---
# 第二阶段：自动微分

对应 Keras 版本的 `GradientTape反向传播.py`。

## §3 autograd 自动求导

### Keras vs PyTorch 梯度计算

```
Keras (tf.GradientTape):
  with tf.GradientTape() as tape:
      logits = model(x)
      loss = loss_fn(y, logits)
  grads = tape.gradient(loss, model.trainable_weights)

PyTorch (autograd):
  outputs = model(x)
  loss = loss_fn(outputs, y)
  loss.backward()  # 自动计算所有参数的梯度，存入 .grad
```

核心区别：
- Keras 需要显式 `with tape:` 包裹前向传播
- PyTorch **自动追踪**所有带 `requires_grad=True` 的张量

In [ ]:
# 3.1 requires_grad 控制梯度追踪

# 默认不追踪
x = torch.tensor([1.0, 2.0, 3.0])
print(f"默认 requires_grad: {x.requires_grad}")  # False

# 开启追踪
x_grad = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
print(f"requires_grad=True: {x_grad.requires_grad}")  # True

# 原地开启
x.requires_grad_(True)
print(f"原地开启: {x.requires_grad}")

In [ ]:
# 3.2 计算图与反向传播

x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(3.0, requires_grad=True)

# z = x² + xy + y²
z = x**2 + x*y + y**2

print(f"z = {z.item()}")  # 4 + 6 + 9 = 19
print(f"z 的 grad_fn: {z.grad_fn}")

# 反向传播
z.backward()

# 查看梯度：dz/dx = 2x + y = 7, dz/dy = x + 2y = 8
print(f"dz/dx = {x.grad.item()}")  # 7.0
print(f"dz/dy = {y.grad.item()}")  # 8.0

In [ ]:
# 3.3 手动实现前向传播 + 反向传播（对应 GradientTape 示例）

# 初始化参数
W = torch.randn(10, 3, requires_grad=True)
b = torch.zeros(3, requires_grad=True)

# 输入
x = torch.randn(5, 10)

# 前向传播
logits = x @ W + b
loss = logits.pow(2).sum()

print(f"前向传播完成")
print(f"logits shape: {logits.shape}")
print(f"loss: {loss.item():.4f}")

# 反向传播
loss.backward()

print(f"\nW 的梯度形状: {W.grad.shape}")
print(f"b 的梯度形状: {b.grad.shape}")
print(f"W 的梯度范数: {W.grad.norm().item():.4f}")

In [ ]:
# 3.4 torch.no_grad() 和 model.eval()

# torch.no_grad()：不计算梯度，节省内存
# 在验证/推理时使用
with torch.no_grad():
    output = torch.randn(5, 10) @ W + b
    print(f"no_grad 模式下 requires_grad: {output.requires_grad}")  # False

# 两者的区别：
# torch.no_grad()   → 不追踪梯度计算
# model.eval()      → 设置层行为（关闭 Dropout，BN 使用全局统计量）
# 两者需要同时使用！

---
# 第三阶段：实战

对应 Keras 版本的 `本地实现图片分类任务.py` 和 `填空版-本地实现图片分类任务.py`。
这里用纯张量操作实现 MNIST 分类，不使用 `nn.Module`。

## §5 纯张量实现 MNIST 分类

### 网络结构
```
Input(784) → Dense(512, relu) → Dropout(0.5) → Dense(10)
```

### 手动实现所有步骤
- 前向传播：手动矩阵乘法 `x @ W + b`
- 激活函数：`torch.relu()`
- 损失计算：交叉熵
- 反向传播：`loss.backward()`
- 参数更新：`W = W - lr * W.grad`

In [ ]:
# 5.1 准备数据
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, TensorDataset

# 加载并预处理
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

full_train = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
full_test = datasets.MNIST(root='./data', train=False, transform=transform)

# 划分验证集
train_size = 50000
val_size = 10000
train_subset, val_subset = torch.utils.data.random_split(full_train, [train_size, val_size])

train_loader = DataLoader(train_subset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=64, shuffle=False)
test_loader = DataLoader(full_test, batch_size=64, shuffle=False)

print(f"训练集: {len(train_subset)} 样本")
print(f"验证集: {len(val_subset)} 样本")
print(f"测试集:  {len(full_test)} 样本")

In [ ]:
# 5.2 手动初始化参数

# Xavier 初始化：适合 relu 激活
torch.manual_seed(42)

W1 = torch.randn(784, 512, requires_grad=True) * (2.0 / 784) ** 0.5
b1 = torch.zeros(512, requires_grad=True)
W2 = torch.randn(512, 10, requires_grad=True) * (2.0 / 512) ** 0.5
b2 = torch.zeros(10, requires_grad=True)

params = [W1, b1, W2, b2]

total_params = sum(p.numel() for p in params)
print(f"总参数量: {total_params}")
# 784*512 + 512 + 512*10 + 10 = 401920 + 512 + 5120 + 10 = 407562

In [ ]:
# 5.3 手动前向传播函数
import torch.nn.functional as F

def forward(x):
    """
    手动实现前向传播。
    Input → Linear(784→512) → ReLU → Linear(512→10)
    """
    x = x.view(-1, 784)  # 展平
    h = x @ W1 + b1      # 第一层
    h = F.relu(h)        # 激活
    logits = h @ W2 + b2  # 输出层
    return logits

In [ ]:
# 5.4 训练循环
epochs = 3
lr = 0.01

for epoch in range(epochs):
    print(f"\n{'='*50}")
    print(f"Epoch {epoch + 1}/{epochs}")
    print(f"{'='*50}")
    
    for step, (images, labels) in enumerate(train_loader):
        # 前向传播
        logits = forward(images)
        loss = F.cross_entropy(logits, labels)
        
        # 反向传播
        loss.backward()
        
        # 手动更新参数（相当于 optimizer.step()）
        with torch.no_grad():
            for p in params:
                p -= lr * p.grad
                p.grad.zero_()  # 清零梯度
        
        if step % 100 == 0:
            print(f"  Step {step:4d}: loss = {loss.item():.4f}")
    
    # 验证
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            logits = forward(images)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    
    print(f"  验证精度: {correct / total:.4f}")

In [ ]:
# 5.5 测试集评估
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        logits = forward(images)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

print(f"测试精度: {correct / total:.4f}")

### 第二章总结

| 概念 | Keras/TF | PyTorch |
|------|----------|---------|
| 张量 | `tf.constant`, `tf.Variable` | `torch.tensor` |
| 自动微分 | `tf.GradientTape` | `loss.backward()` |
| 梯度访问 | `tape.gradient(loss, weights)` | `param.grad` |
| 参数更新 | `optimizer.apply_gradients()` | `param -= lr * param.grad` |
| 不计算梯度 | 不在 tape 中 | `torch.no_grad()` |
| 与 NumPy 互操作 | `tensor.numpy()` | `tensor.numpy()` / `torch.from_numpy()` |
| 设备 | `tf.device()` | `tensor.to(device)` |

---
**下一步学习**：打开 `第三章-PyTorch入门/PyTorch核心API全解.ipynb` 学习如何使用 `nn.Module` 和 `DataLoader` 构建更结构化的模型。